# A Theorem Prover for First-Order Logic without Equality

This notebook implements a resolution-based theorem prover in TypeScript. 

We need the parser for first order formulas, hence we import it. Formulas are represented as nested tuples. In our TypeScript parser, variables start with an uppercase letter, and predicate and function symbols start with a lowercase letter.

The resolution calculus works with clauses. The file `FOL-CNF.ts` implements the function `normalize(f)` that turns a formula `f` into a set of clauses.

The module `Unification` implements unification via the algorithm of Martelli and Montanari.

In [ ]:
import { parseFormula as parse } from "./FOL-Parser";
import { normalize } from "./FOL-CNF";
import { unify, Term, Substitution } from "./Unification";
import { Tuple, RecursiveSet as Set, RecursiveMap as Map } from "recursive-set";

## Domain Types

Because `Tuple` encapsulates its elements, we strictly type our AST structures to match the output of the CNF normalizer.

In [ ]:
type FunctionSymbol  = string;
type PredicateSymbol = string;

type TupleTerm = string | Tuple<[FunctionSymbol, ...TupleTerm[]]>;
type TupleAtom = Tuple<['⚛️', PredicateSymbol, ...TupleTerm[]]>;
type Literal   = TupleAtom | Tuple<['¬', TupleAtom]>;
type Clause    = Set<Literal>;

## Safely Unpacking Iterables

The function `isIterable` acts as a runtime type guard to verify if an unknown object implements the iterable protocol, allowing safe extraction without casts.

In [ ]:
function isIterable(obj: unknown): obj is Iterable<unknown> {
    return typeof obj == 'object' && obj !== null && Symbol.iterator in obj;
}

The function `extractArray` safely converts an iterable object into a standard array to allow strictly checked indexed access.

In [ ]:
function extractArray(obj: unknown): unknown[] {
    if (isIterable(obj)) {
        return Array.from(obj);
    }
    return [];
}

The function `isTupleTerm` acts as a structural type guard to confirm whether an object mathematically represents a valid term (either a variable string or a `Tuple` application).

In [ ]:
function isTupleTerm(obj: unknown): obj is TupleTerm {
    return typeof obj == 'string' || obj instanceof Tuple;
}

The function `isTupleAtom` verifies if an object represents an atomic formula, specifically checking for the unique `'⚛️'` tag implemented by our parser.

In [ ]:
function isTupleAtom(obj: unknown): obj is TupleAtom {
    if (!(obj instanceof Tuple)) return false;
    const arr = extractArray(obj);
    return arr[0] == '⚛️';
}

Given a literal `l`, the function `isNegativeLiteral` checks if the literal represents a negated atomic formula.

In [ ]:
function isNegativeLiteral(l: Literal): l is Tuple<['¬', TupleAtom]> {
    const arr = extractArray(l);
    return arr[0] === '¬';
}

Given a literal `l`, the function `atomOf` safely strips away any negation, returning the underlying positive atomic formula.

In [ ]:
function atomOf(l: Literal): TupleAtom {
    if (isNegativeLiteral(l)) {
        const arr = extractArray(l);
        const atom = arr[1];
        if (isTupleAtom(atom)) {
            return atom;
        }
        throw new Error("Malformed negative literal.");
    }
    return l;
}

The function `unpackTerm` safely destructures a complex term into its governing function symbol and an array of its argument terms.

In [ ]:
function unpackTerm(t: TupleTerm): { f: string, args: TupleTerm[] } {
    if (typeof t === 'string') throw new Error("Cannot unpack a string variable.");
    const arr = extractArray(t);
    const f = typeof arr[0] == 'string' ? arr[0] : "";
    const args: TupleTerm[] = [];
    for (let i = 1; i < arr.length; i++) {
        const item = arr[i];
        if (isTupleTerm(item)) args.push(item);
    }
    return { f, args };
}

The function `unpackAtom` safely destructures an atomic formula into its internal tag, its predicate symbol, and its arguments.

In [ ]:
function unpackAtom(a: TupleAtom): { tag: string, pred: string, args: TupleTerm[] } {
    const arr  = extractArray(a);
    const tag  = typeof arr[0] == 'string' ? arr[0] : "";
    const pred = typeof arr[1] == 'string' ? arr[1] : "";
    const args: TupleTerm[] = [];
    for (let i = 2; i < arr.length; i++) {
        const item = arr[i];
        if (isTupleTerm(item)) args.push(item);
    }
    return { tag, pred, args };
}

## Variable Extraction and Substitution

The function `collectVariablesTerm` computes the variables occurring in a term.

In [ ]:
function collectVariablesTerm(t: TupleTerm): Set<string> {
    if (typeof t == 'string') {
        if (/^[A-Z][a-zA-Z0-9_]*$/.test(t)) {
            return new Set(t);
        }
        return new Set<string>();
    }
    const unpacked = unpackTerm(t);
    return unpacked.args.reduce((acc, arg) => acc.union(collectVariablesTerm(arg)), new Set<string>());
}

The function `collectVariablesLit` computes the variables occurring in a literal.

In [ ]:
function collectVariablesLit(l: Literal): Set<string> {
    const atom = atomOf(l);
    const unpacked = unpackAtom(atom);
    return unpacked.args.reduce((acc, arg) => acc.union(collectVariablesTerm(arg)), new Set<string>());
}

Given a clause `C`, the function `collectVariablesClause(C)` computes the set of all variables occurring in `C`.

In [ ]:
function collectVariablesClause(c: Clause): Set<string> {
    return Array.from(c).reduce((acc, lit) => acc.union(collectVariablesLit(lit)), new Set<string>());
}

The function `applySubstTerm` structurally applies a variable renaming map (substitution) to a specific term.

In [ ]:
const ascii_uppercase = "ABCDEFGHIJKLMNOPQRSTUVWXYZ".split("");

function applySubstTerm(t: TupleTerm, sigma: Map<string, string>): TupleTerm {
    if (typeof t == 'string') {
        const mapped = sigma.get(t);
        return mapped !== undefined ? mapped : t;
    }
    const unpacked = unpackTerm(t);
    return new Tuple(unpacked.f, ...unpacked.args.map(arg => applySubstTerm(arg, sigma)));
}

The function `applySubstAtom` structurally applies a variable renaming map to an atomic formula, retaining strict type references.

In [ ]:
function applySubstAtom(a: TupleAtom, sigma: Map<string, string>): TupleAtom {
    const unpacked = unpackAtom(a);
    return new Tuple('⚛️', unpacked.pred, ...unpacked.args.map(arg => applySubstTerm(arg, sigma)));
}

The function `applySubstLit` applies a substitution mapping to a literal, wrapping the modified atom back into a negative tuple if it was negated.

In [ ]:
function applySubstLit(l: Literal, sigma: Map<string, string>): Literal {
    if (isNegativeLiteral(l)) {
        const atom = atomOf(l);
        return new Tuple('¬', applySubstAtom(atom, sigma));
    }
    return applySubstAtom(l, sigma);
}

The function `renameVariables(f, g)` takes two clauses `f` and `g` and renames the variables in the clauses `f` so that they are different from the variables occurring in `g`.

In [ ]:
function renameVariables(f: Clause, g: Clause): Clause {
    const oldVars = Array.from(collectVariablesClause(f));
    const gVars = collectVariablesClause(g);
    const freshVars = ascii_uppercase.filter(v => !gVars.has(v));

    const sigma = new Map<string, string>();
    oldVars.forEach((v, i) => {
        if (i < freshVars.length) {
            sigma.set(v, freshVars[i]);
        }
    });

    const newClause = new Set<Literal>();
    for (const lit of f) {
        newClause.add(applySubstLit(lit, sigma));
    }
    return newClause;
}

## Type-Safe Unification Bridge

The function `tupleTermToTerm` translates a typed local `TupleTerm` structure into the untagged array tuple format expected by the `Unification.ts` module.

In [ ]:
function tupleTermToTerm(t: TupleTerm): Term {
    if (typeof t == 'string') return t;
    const unpacked = unpackTerm(t);
    return [unpacked.f, ...unpacked.args.map(tupleTermToTerm)];
}

The function `atomToTerm` translates a typed local `TupleAtom` (stripping away its logical type-tags) into an untagged array tuple.

In [ ]:
function atomToTerm(a: TupleAtom): Term {
    const unpacked = unpackAtom(a);
    return [unpacked.pred, ...unpacked.args.map(tupleTermToTerm)];
}

The function `uTermToTupleTerm` parses the structural array tuple returned by the unification algorithm back into an immutable, `recursive-set` backed `TupleTerm`.

In [ ]:
function uTermToTupleTerm(t: Term): TupleTerm {
    if (typeof t == 'string') return t;
    const [f, ...args] = t;
    return new Tuple(f, ...args.map(uTermToTupleTerm));
}

The function `applyMuTerm` resolves a variable inside a term against the computed unifier map `mu`.

In [ ]:
function applyMuTerm(t: TupleTerm, mu: Substitution): TupleTerm {
    if (typeof t == 'string') {
        const mapped = mu.get(t);
        return mapped !== undefined ? uTermToTupleTerm(mapped) : t;
    }
    const unpacked = unpackTerm(t);
    return new Tuple(unpacked.f, ...unpacked.args.map(arg => applyMuTerm(arg, mu)));
}

The function `applyMuAtom` resolves a computed unifier against all arguments within an atomic formula.

In [ ]:
function applyMuAtom(a: TupleAtom, mu: Substitution): TupleAtom {
    const unpacked = unpackAtom(a);
    return new Tuple('⚛️', unpacked.pred, ...unpacked.args.map(arg => applyMuTerm(arg, mu)));
}

The function `applyMuLit` resolves a computed unifier against a literal, reconstructing the logical negation if necessary.

In [ ]:
function applyMuLit(l: Literal, mu: Substitution): Literal {
    if (isNegativeLiteral(l)) {
        const atom = atomOf(l);
        return new Tuple('¬', applyMuAtom(atom, mu));
    }
    return applyMuAtom(l, mu);
}

## A Calculus for First Order Logic

The resolution rule is an inference rule that is defined as follows: If
 * $C_1$ and $C_2$ are clauses from first order logic,
 * $p(s_1,\cdots,s_n)$ and $p(t_1,\cdots,t_n)$ are atomic formulas,
 * the syntactical equation $p(s_1,\cdots,s_n) \doteq p(t_1,\cdots,t_n)$ is solvable and
     $$ \mu = \mathtt{mgu}\bigl(p(s_1,\cdots,s_n), p(t_1,\cdots,t_n)\bigr), $$
then
$$\frac{C_1 \cup\{ p(s_1,\cdots,s_n)\} \quad\quad \{\neg p(t_1,\cdots,t_n)\} \cup C_2}{
                 C_1\mu \cup C_2\mu} 
$$
is an application of the resolution rule.

Given a two clauses `C1` and `C2`, the function `resolve(C1, C2)` computes a set of all clauses that can be inferred from `C1` and `C2` by applying the resolution rule.

In [ ]:
function resolve(C1: Clause, C2: Clause): Set<Clause> {
    const C2New  = renameVariables(C2, C1);
    const result = new Set<Clause>();

    for (const L1 of C1) {
        for (const L2 of C2New) {
            if (isNegativeLiteral(L1) != isNegativeLiteral(L2)) {
                const t1 = atomToTerm(atomOf(L1));
                const t2 = atomToTerm(atomOf(L2));
                const mu = unify(t1, t2);
                if (mu) {
                    const C1Rem = Array.from(C1   ).filter(l => !l.equals(L1));
                    const C2Rem = Array.from(C2New).filter(l => !l.equals(L2));
                    
                    const resolventArr     = [...C1Rem, ...C2Rem];
                    const resolventApplied = resolventArr.map(l => applyMuLit(l, mu));
                    
                    result.add(new Set(...resolventApplied));
                }
            }
        }
    }
    return result;
}

## Factorization

A calculus which only contains the resolution rule is not complete. We also need the factorization rule. If
1. $C$ is a clause from first order logic,
2. $p(s_1,\cdots,s_n)$ and $p(t_1,\cdots,t_n)$ are atomic formulas,
3. the syntactical equation $p(s_1,\cdots,s_n)  \doteq p(t_1,\cdots,t_n)$ is solvable and 
     $$\mu = \mathtt{mgu}\bigl(p(s_1,\cdots,s_n), p(t_1,\cdots,t_n)\bigr), $$
then both 
$\displaystyle \frac{C \cup \bigl\{p(s_1,\cdots,s_n),\, p(t_1,\cdots,t_n)\bigl\}}{C\mu \cup \bigl\{p(s_1,\cdots,s_n)\mu\bigr\} } $ 
and 
$\displaystyle \frac{C \cup \bigl\{ \neg p(s_1,\cdots,s_n),\, \neg p(t_1,\cdots,t_n)\bigl\}}{C\mu \cup \bigl\{\neg p(s_1,\cdots,s_n)\mu\bigr\} }
$
are applications of the factorization rule.

The function `factorize(C)` takes a clause `C` from first order logic and computes all clauses that can be derived from `C` via factorization.

In [ ]:
function factorize(C: Clause): Set<Clause> {
    const result = new Set<Clause>();
    const arrC = Array.from(C);
    for (let i = 0; i < arrC.length; i++) {
        for (let j = i + 1; j < arrC.length; j++) {
            const L1 = arrC[i];
            const L2 = arrC[j];
            if (isNegativeLiteral(L1) == isNegativeLiteral(L2)) {
                const t1 = atomToTerm(atomOf(L1));
                const t2 = atomToTerm(atomOf(L2));
                const mu = unify(t1, t2);
                if (mu) {
                    const applied = arrC.map(l => applyMuLit(l, mu));
                    result.add(new Set(...applied));
                }
            }
        }
    }
    return result;
}

## Automatic Theorem Proving

Given a set of clauses `Clauses`, the function `infere(Clauses)` returns an array of all possible clauses that result from the resolution of two clauses or the factorization of a single clause within the set.

In [ ]:
type Reason = Tuple<[Clause]> | Tuple<[Clause, Clause]>;

In [ ]:
function infere(Clauses: Set<Clause>): Array<[Clause, Reason]> {
    const result: Array<[Clause, Reason]> = [];
    const arrClauses = Array.from(Clauses);

    for (const C1 of arrClauses) {
        for (const C2 of arrClauses) {
            const res = resolve(C1, C2);
            for (const C of res) {
                result.push([C, new Tuple(C1, C2)]);
            }
        }
        const facts = factorize(C1);
        for (const C of facts) {
            result.push([C, new Tuple(C1)]);
        }
    }
    return result;
}

The function `saturate(Cs)` takes a set of clauses `Cs` as input and tries to infer the empty clause. If it is not possible to infer the empty clause, the function runs until memory is exhausted.

In [ ]:
function saturate(Cs: Set<Clause>): Map<Clause, Reason> {
    const Clauses = new Set(...Array.from(Cs));
    let cnt = 1;
    const Reasons = new Map<Clause, Reason>();
    const emptyClause = new Set<Literal>();

    while (!Clauses.has(emptyClause)) {
        let added = false;
        const inferences = infere(Clauses);
        for (const [C, R] of inferences) {
            if (!Clauses.has(C)) {
                Reasons.set(C, R);
                Clauses.add(C);
                added = true;
            }
        }
        console.log(`cnt = ${cnt}, number of clauses: ${Clauses.size}`);
        if (!added) break;
        cnt += 1;
    }
    return Reasons;
}

## Proof Reconstruction and Printing Helpers

The function `stringifyTerm` converts a typed local `TupleTerm` into a flat string representation suitable for terminal output.

In [ ]:
function stringifyTerm(t: TupleTerm): string {
    if (typeof t == 'string') return t;
    const unpacked = unpackTerm(t);
    if (unpacked.args.length == 0) return unpacked.f;
    return `${unpacked.f}(${unpacked.args.map(stringifyTerm).join(', ')})`;
}

The function `stringifyAtom` converts a typed atomic formula tuple into a string.

In [ ]:
function stringifyAtom(a: TupleAtom): string {
    const unpacked = unpackAtom(a);
    if (unpacked.args.length == 0) return unpacked.pred;
    return `${unpacked.pred}(${unpacked.args.map(stringifyTerm).join(', ')})`;
}

The function `stringifyLiteral` renders a literal object to a string, conditionally inserting the negation symbol if required.

In [ ]:
function stringifyLiteral(l: Literal): string {
    if (isNegativeLiteral(l)) {
        const atom = atomOf(l);
        return `¬${stringifyAtom(atom)}`;
    }
    return stringifyAtom(l);
}

The function `stringifyClause` maps over a clause set and formats it securely within brackets as a comma-separated list.

In [ ]:
function stringifyClause(C: Clause): string {
    if (C.size == 0) return "{}";
    const arr = Array.from(C).map(stringifyLiteral);
    return `{${arr.join(', ')}}`;
}

The function `updateProof` merges a generated proof segment `P2` into `P1` while guaranteeing lines are not duplicated.

In [ ]:
function updateProof(P1: string[], P2: string[]): string[] {
    const result = [...P1];
    for (const line of P2) {
        if (!result.includes(line)) result.push(line);
    }
    return result;
}

Given a dictionary `Reasons` and a clause `clause`, the function `constructProof` traverses backwards along the reason graph and returns a text-based proof array leading up to `clause`.

In [ ]:
function constructProof(clause: Clause, Reasons: Map<Clause, Reason>): string[] {
    const reason = Reasons.get(clause);
    if (reason === undefined) {
        return [`Axiom:       ${stringifyClause(clause)}`];
    }
    
    const arr = extractArray(reason);

    if (arr.length == 1) {
        const C = arr[0];
        if (C instanceof Set) {
            const Proof = constructProof(C, Reasons);
            Proof.push(`Factorization: ${stringifyClause(C)} \n⊢            ${stringifyClause(clause)}`);
            return Proof;
        }
    }
    if (arr.length == 2) {
        const C1 = arr[0];
        const C2 = arr[1];
        if (C1 instanceof Set && C2 instanceof Set) {
            const ProofC1 = constructProof(C1, Reasons);
            const ProofC2 = constructProof(C2, Reasons);
            const Proof = updateProof(ProofC1, ProofC2);
            Proof.push(`Resolution:  ${stringifyClause(C1)},\n             ${stringifyClause(C2)}  \n⊢            ${stringifyClause(clause)}`);
            return Proof;
        }
    }
    return [];
}

## Testing with the Red Dragons

According to Uwe Schöning, the theory of red dragons is given by the following axioms:

1. Every dragon is happy if all its children can fly.
2. All red dragons can fly.
3. The children of red dragons are themselves red.

We will show that these axioms imply that all red dragons are happy. To this end, the formula stating that all red dragons can fly is negated. Then we will show that the set consisting of the negated formula together with the axioms is inconsistent.

In [ ]:
const s1 = '∀X:(∀Y:(child(Y, X) → canFly(Y)) → happy(X))';
const s2 = '∀X:(red(X) → canFly(X))';
const s3 = '∀X:(red(X) → ∀Y:(child(Y, X) → red(Y)))';
const s4 = '¬∀X:(red(X) → happy(X))';

const c1 = Array.from(normalize(parse(s1)));
const c2 = Array.from(normalize(parse(s2)));
const c3 = Array.from(normalize(parse(s3)));
const c4 = Array.from(normalize(parse(s4)));

const Clauses = new Set<Clause>(...c1, ...c2, ...c3, ...c4);

console.log("Saturating...");
const Reasons = saturate(Clauses);
const Proof = constructProof(new Set<Literal>(), Reasons);

Proof.forEach(line => console.log(line));